# Virtual assistant: task classification

Dans ce TP, on va entraîner un modèle "requête -> pipeline à lancer" (les pipelines possibles sont "question_rag" et "send_message")</br>
On est sur un problème de classification classique, mais le transfer learning permet vite d'avoir de bons résultats, malgré un faible jeu de données.

L'erreur que j'ai vu le plus souvent est d'entraîner le full modèle DistilBert (ou autre). Notre dataset est très faible, on ne peut pas entraîner un réseau de neurones complet. Il **faut** freeze des layers et n'en apprendre que quelque-unes. Sinon, notre modèle va over-fitter et ne généralisera pas.<br/>


In [1]:
import pandas as pd
import transformers

## Data loading

In [2]:
df_train = pd.read_csv("../data/raw/question_classif.csv")
df_train.head()

,question,label_text,label
0,What are the recommended prerequisites for the...,question_rag,0
1,Does the cybersecurity course cover intrusion ...,question_rag,0
2,How can I enroll in the Python course?,question_rag,0
3,What are the main basic concepts covered in th...,question_rag,0
4,Does the React course include practical projec...,question_rag,0


In [28]:
df_test = pd.read_csv("../data/raw/question_classif_test.csv")
df_test.head()

,question,label_text,label
0,How does the React course address accessibilit...,question_rag,0
1,Does the introduction to machine learning cour...,question_rag,0
2,Does the Python course include modules on web ...,question_rag,0
3,What are the main challenges that businesses f...,question_rag,0
4,How does the SQL course deal with distributed ...,question_rag,0


In [4]:
sentences_train = df_train["question"]
labels_train = df_train["label"]

sentences_test = df_test["question"]
labels_test = df_test["label"]

In [11]:
id2label = {row.label: row.label_text for _, row in df_train[["label_text", "label"]].drop_duplicates().iterrows()}
label2id = {label: id for id, label in id2label.items()}

## Model loading

In [12]:
model_name = "distilbert-base-cased"

In [13]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name, add_prefix_space=True)

In [14]:
tokenized_train = tokenizer(list(sentences_train))
tokenized_train["label"] = labels_train

In [15]:
tokenized_test = tokenizer(list(sentences_test))
tokenized_test["label"] = labels_test

In [16]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2, id2label=id2label, label2id=label2id
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-cased and are newly initialized: ['pre_classifier.weight', 'classifier.bias', 'classifier.weight', 'pre_classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [17]:
from datasets import Dataset

dataset_train = Dataset.from_dict(tokenized_train)
dataset_test = Dataset.from_dict(tokenized_test)

In [18]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Freeze layers

In [19]:
for name, param in model.base_model.named_parameters():
  param.requires_grad = False

for name, param in model.base_model.named_parameters():
    if any(txt in name for txt in ["layer.5.ffn.lin1", "layer.5.ffn.lin2", "layer.5.output_layer_norm"]):
        param.requires_grad = True

## Metrics

In [20]:
import evaluate

accuracy = evaluate.load("accuracy")

In [21]:
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

# Train

In [22]:
training_args = TrainingArguments(
    output_dir="classify_task",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=20,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.675542,0.609756
2,No log,0.656603,0.951220
3,No log,0.638964,0.951220
4,No log,0.622315,1.000000
5,No log,0.606697,1.000000
6,No log,0.591337,0.975610
7,No log,0.576977,1.000000
8,No log,0.563944,0.975610
9,No log,0.549815,1.000000
10,No log,0.535472,1.000000


TrainOutput(global_step=140, training_loss=0.5577900477818081, metrics={'train_runtime': 32.0016, 'train_samples_per_second': 61.247, 'train_steps_per_second': 4.375, 'total_flos': 11189355580224.0, 'train_loss': 0.5577900477818081, 'epoch': 20.0})

In [23]:
import huggingface_hub

In [24]:
token_hugging_face = None  # Insert your token here
huggingface_hub.login(token_hugging_face)

Token will not been saved to git credential helper. Pass `add_to_git_credential=True` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /home/arnaud/.cache/huggingface/token
Login successful


In [25]:
tokenizer.push_to_hub("nlp_esgi_td5_classification")

CommitInfo(commit_url='https://huggingface.co/foucheta/nlp_esgi_td5_classification/commit/31051fe9a135db938a895e696f0c7a96e5b9bb81', commit_message='Upload tokenizer', commit_description='', oid='31051fe9a135db938a895e696f0c7a96e5b9bb81', pr_url=None, pr_revision=None, pr_num=None)

In [26]:
model.push_to_hub("nlp_esgi_td5_classification")

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/foucheta/nlp_esgi_td5_classification/commit/e1ce91519c1cd7faf900adc6f27124d900e995fa', commit_message='Upload DistilBertForSequenceClassification', commit_description='', oid='e1ce91519c1cd7faf900adc6f27124d900e995fa', pr_url=None, pr_revision=None, pr_num=None)